# Tugas Besar 2 IF3170 Inteligensi Artifisial

*Klasifikasi pembatalan pemesanan hotel (`is_canceled`).*

## IF3170_G08_Teazzi

- 13524034 - Jason Edward Salim
- 13524064 - Stefani Angeline Oroh
- 13524067 - Bryan Pratama Putra Hendra
- 13524079 - Angelina Andra Alanna
- 13524106 - Michael James Liman

## Table of Contents (CRISP-DM)

1. [**Business Understanding**](#1)
2. [**Data Understanding**](#2)
3. [**Data Preparation**](#3)
4. [**Modeling**](#4)
5. [**Evaluation**](#5)
6. [**Deployment**](#6)

## Initialization

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd() if Path.cwd().name == "src" else Path.cwd() / "src"))

from config import Settings, seed_everything
from eda import types, missing, duplicates, pie, bar, histogram, boxplot, iqr_outliers, target_rate, correlation
from preprocessing import load_data, split, drop_duplicates, Preprocessor, Pipeline
from validation import fit, holdout, kfold, compare, tune, best, importance, confusion, report, misclassified, submit
from models import KNN, LogisticRegression, SVM, DTL, BayesNetwork

seed_everything()

In [ ]:
train, test = load_data()
pd.Series({"train": len(train), "test": len(test)})

---

# Business Understanding <a name="1"></a>

---

Memprediksi apakah sebuah pemesanan hotel akan dibatalkan (`is_canceled = 1`) atau tidak (`is_canceled = 0`). Metrik evaluasi: macro F1-score.

> TODO: konteks bisnis dan dampak false positive / false negative.

---

# Data Understanding <a name="2"></a>

---

## Data Overview

In [ ]:
train

In [ ]:
display(types(train))
display(types(test))

In [ ]:
missing(train, test)

In [ ]:
duplicates(train, test)

## Target Distribution

In [ ]:
pie(train)
bar(train)

## Feature Analysis

In [ ]:
num_col = train.select_dtypes("number").columns.drop(Settings.TARGET)
cat_col = train.columns.difference(num_col).drop([Settings.ID, Settings.TARGET])

display(train[num_col].describe().T)
display(train[cat_col].describe().T)

In [ ]:
histogram(train, num_col)
boxplot(train, num_col)

In [ ]:
iqr_outliers(train, num_col)

In [ ]:
target_rate(train, list(cat_col) + ['special_requests', 'arrival_month', 'is_returning_guest'])

In [ ]:
correlation(train)

#### Insights

> TODO

---

# Data Preparation <a name="3"></a>

---

Data dibagi sebelum preprocessing. Seluruh langkah di-fit hanya pada `X_train`, lalu diterapkan ke `X_train`, `X_val`, dan `X_test`. Logika ada di `src/preprocessing.py` (fitur baru di `engineer`).

Tiap model memakai preprocessing yang paling cocok untuknya, dipilih lewat macro F1 5-fold CV: `default` untuk BN, `svm` untuk SVM, `tree` (tanpa scaling dan capping) untuk DTL, `knn` (dengan fitur baru dan penanda missing `special_requests`) untuk KNN, dan `lr` (one-hot dari bin kuantil) untuk LR. Preprocessing `svm` menambah dua kelompok fitur berbasis label. Pertama, `near_count`/`near_rate` (dan `near2_*` dengan toleransi harga lebih longgar): banyaknya baris latih yang tampak seperti pesanan grup yang sama (tanggal kedatangan, lama menginap, channel, kamar, dan lainnya sama; lead time selisih ≤ 10 hari; harga selisih ≤ 3% atau ≤ 10%) beserta tingkat pembatalannya. Pasangan seperti ini sama labelnya pada 95% kasus. Kedua, target encoding `te_*`: tingkat pembatalan baris latih dengan kunci yang sama, misalnya bulan kedatangan × pita lead time. Untuk baris latih, keduanya dihitung out-of-fold sehingga sebuah baris tidak pernah melihat labelnya sendiri. Kunci dengan pita harga (`price10`, harga per 10%) ikut dipakai. Macro F1 5-fold SVM naik dari 0,838 menjadi 0,868.

`X_trains`, `X_vals`, dan `X_tests` memetakan nama model ke data hasil preprocessing-nya.

In [ ]:
X_train, X_val, y_train, y_val = split(train)
X_test = test.drop(columns=[Settings.ID])

X_train, y_train = drop_duplicates(X_train, y_train)
pd.Series({"X_train": len(X_train), "X_val": len(X_val), "X_test": len(X_test)})

In [ ]:
CAP = ["days_before_arrival", "nightly_rate_idr"]
ORDINAL = {
    "meal_package": ["Room Only", "Breakfast", "Half Board", "Full Board"],
    "room_category": [f"Type {c}" for c in "ABCDEFG"],
}
# Features from preprocessing.py that only helped KNN in 5-fold CV, dropped for the other models
NEW_FEATURES = ["arrival_weekday", "booking_month", "booking_weekday", "price_per_guest", "nightly_rate_idr_ratio", "days_before_arrival_ratio"]
# Target encodings for the SVM: lead_band = lead time bands, lead20 = 20-day lead bins, price10 = 10%-wide price bins,
# nights = weekend + weekday nights
TARGET_ENCODE = [
    ("arrival_year", "arrival_month", "lead_band"), ("nights", "booking_channel", "lead_band"), ("date", "lead_band"),
    ("week", "lead_band"), ("date", "booking_channel"), ("room_category", "lead_band"), ("special_requests", "lead_band"),
    ("booking_channel", "room_category"), ("arrival_year", "arrival_month", "booking_channel"), ("date", "nights"),
    ("meal_package", "lead_band"), ("num_adults", "booking_channel", "lead_band"), ("lead20", "special_requests", "booking_channel"),
    ("lead20", "arrival_year", "arrival_month"), ("lead_band", "special_requests", "booking_channel", "arrival_year"),
    ("lead20", "room_category", "special_requests"), ("price10", "lead_band"), ("price10", "room_category", "arrival_year"),
    ("price10", "special_requests", "booking_channel"), ("price10", "arrival_year", "arrival_month"),
]
BINNED = ["days_before_arrival", "nightly_rate_idr", "arrival_month", "date_rate", "special_requests", "total_nights",
          "arrival_weekday", "booking_month", "price_per_guest", "num_adults"]

preps = {
    "default": Preprocessor(
        zero_as_missing=[],                                   # e.g. ["nightly_rate_idr"]
        num_strategy="median",
        cap=CAP,
        ordinal=ORDINAL,
        scaler="standard",                                    # 'standard' | 'minmax' | 'robust' | None
        drop=NEW_FEATURES,
    ),
    # + group-booking neighbours (near_*) and out-of-fold target encodings (te_*)
    "svm": Preprocessor(cap=CAP, ordinal=ORDINAL, flag_missing=["special_requests"], near_dup=True,
                        near_tols=[(0, 10, 0.03), (0, 10, 0.10)], target_encode=TARGET_ENCODE, te_smoothing=5),
    "tree": Preprocessor(ordinal=ORDINAL, scaler=None, drop=NEW_FEATURES),                     # tree splits ignore scale
    "knn": Preprocessor(cap=CAP, ordinal=ORDINAL, scaler="robust", smoothing=5),
    "lr": Preprocessor(cap=CAP, flag_missing=["special_requests"], binned=BINNED, n_bins=20),   # bins let a linear model bend
}
uses = {"svm": "svm", "bn": "default", "bn_kdb": "default", "dtl": "tree", "knn": "knn", "lr": "lr",
        "sk_svm": "svm", "sk_nb": "default", "sk_dtl": "tree", "sk_knn": "knn", "sk_lr": "lr"}

prepared = {name: (p.fit_transform(X_train, y_train), p.transform(X_val), p.transform(X_test)) for name, p in preps.items()}
X_trains, X_vals, X_tests = ({model: prepared[prep][i] for model, prep in uses.items()} for i in range(3))
X_train, X_val, X_test = prepared["default"]
X_train.head()

#### Keputusan

> TODO: justifikasi imputasi, capping, encoding, scaling.

---

# Modeling <a name="4"></a>

---

Model from scratch ada di `src/models/` (implementasi `fit`/`predict`). Model sklearn dipakai sebagai pembanding. `METHOD` menentukan penanganan class imbalance (`None`, `'smote'`, `'undersample'`), hanya diterapkan pada data latih.

In [ ]:
METHOD = None

knn_config = {
    "k": 91,
    "metric": "manhattan",
    "vote_weights": "inverse_square",
    "feature_weights": {
        "meal_package": 0.2,
        "room_category": 0.25,
        "date_rate": 0.5,
        "month_sin": 0,
        "month_cos": 0,
        "doy_sin": 0,
        "doy_cos": 0,
    },
    "threshold": 0.45,
}

models = {
    "knn": KNN(**knn_config),
    "lr": LogisticRegression(lr=0.5, n_iter=3000),
    "svm": SVM(kernel="rbf", C=2, gamma=0.025),
    "dtl": DTL(criterion="entropy", max_depth=10),
    "bn": BayesNetwork(),
    "bn_kdb": BayesNetwork(n_bins=5, k=10),
}

In [ ]:
from sklearn.linear_model import LogisticRegression as SkLogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

# Same settings and data as the from-scratch models
sk_models = {
    "sk_knn": Pipeline(KNeighborsClassifier(5, metric="manhattan"), select=12, ranker=ranker),
    "sk_lr": SkLogisticRegression(max_iter=3000),
    "sk_svm": SVC(kernel="rbf", C=2, gamma=0.025),
    "sk_dtl": DecisionTreeClassifier(criterion="entropy", max_depth=10, random_state=Settings.SEED),
    "sk_nb": GaussianNB(),
}

## SVM Feature Selection

SVM diselesaikan dalam bentuk dual dengan SMO, sehingga untuk kernel linear w = Σ αᵢyᵢxᵢ. Fitur diranking berdasarkan |w| SVM linear yang dilatih pada seluruh fitur, lalu `select` fitur teratas dipakai untuk melatih ulang SVM. Model final memakai kernel RBF, yang tidak memiliki w eksplisit.

In [ ]:
svm_fitted = fit(Pipeline(SVM(kernel="linear", C=0.1), select=10), X_train, y_train)
pd.Series(X_train.columns[svm_fitted.keep], name="svm selected features")

---

# Evaluation <a name="5"></a>

---

## Model Comparison

Holdout pada `X_val` dan K-Fold pada `X_train` dengan macro F1-score.

In [ ]:
all_models = {**models, **sk_models}
compare(holdout(all_models, X_trains, y_train, X_vals, y_val, METHOD), kfold(all_models, X_trains, y_train, METHOD))

## Hyperparameter Tuning

KNN dituning pada `k={71, 91, 111}` dan `threshold={0.42, 0.44, 0.45, 0.46, 0.48}` dengan macro F1. Bobot fitur dan preprocessing mengikuti `knn_config` serta konfigurasi KNN di atas.

In [ ]:
KNN_THRESHOLDS = [0.42, 0.44, 0.45, 0.46, 0.48]
grids = {
    "knn": {"k": [71, 91, 111], "threshold": KNN_THRESHOLDS},
    "lr": {"lr": [0.1, 0.5], "n_iter": [1000, 3000]},
    "svm": {"kernel": ["rbf"], "C": [1, 2], "gamma": [0.025, 0.035]},
    "dtl": {"max_depth": [8, 10, 12], "min_samples_split": [2, 20]},
    "bn": {"n_bins": [3, 5, 10]},
    "bn_kdb": {"k": [6, 8, 10]},
}

grid_tables = {name: tune(models[name], grid, X_trains[name], y_train, METHOD) for name, grid in grids.items()}
tuned_models = {name: best(models[name], table) for name, table in grid_tables.items()}
grid_tables["knn"].head()

In [ ]:
ranking = compare(
    holdout(tuned_models, X_trains, y_train, X_vals, y_val, METHOD),
    kfold(tuned_models, X_trains, y_train, METHOD),
)
ranking

## Error Analysis

In [ ]:
# Pakai model from scratch terbaik, selama belum ada yang diimplementasikan, pakai baseline sklearn
if ranking.empty:
    ranking = compare(holdout(sk_models, X_trains, y_train, X_vals, y_val, METHOD), kfold(sk_models, X_trains, y_train, METHOD))
    tuned_models = {**tuned_models, **sk_models}

best_name = ranking.index[0]
best_model = tuned_models[best_name]
best_name, best_model

In [ ]:
confusion(tuned_models, X_trains, y_train, X_vals, y_val, METHOD)

In [ ]:
report(tuned_models, X_trains, y_train, X_vals, y_val, METHOD)

In [ ]:
wrong_rows = misclassified(best_model, train, X_trains[best_name], y_train, X_vals[best_name], y_val, METHOD)
print(f"{len(wrong_rows)} dari {len(X_val)} baris salah")
wrong_rows["predicted"].map({1: "false positive", 0: "false negative"}).value_counts()

In [ ]:
importance(sk_models["sk_dtl"], X_trains["sk_dtl"], y_train).head(10)

In [ ]:
from models.dtl import plot_tree

# top_n = number of split levels from the root drawn in the image
dtl_fitted = fit(tuned_models["dtl"], X_trains["dtl"], y_train, METHOD)
fig = plot_tree(dtl_fitted, list(X_trains["dtl"].columns), top_n=3, path=Settings.OUTPUT_DIR / "dtl_tree.png")
fig

#### Insights

> TODO

## Bonus Bayesian Network

Bonus pada Bayesian Network: **TAN** dan **KDB** (optimasi struktur), **graf**, dan **cost-sensitive classification**. Kodenya ada di `src/models/bayes_network.py` (parameter `tree`, `k`, `cost_fp`, `cost_fn`) dan `src/models/bayes_graph.py`. Model utama di bagian Modeling tetap Naive Bayes (`BayesNetwork()`).

### TAN (Tree-Augmented Naive Bayes)

Naive Bayes mengasumsikan fitur saling independen jika kelas diketahui, sehingga fitur yang berkaitan (misalnya `weekday_nights` dan `weekend_nights`) dihitung dua kali dan probabilitasnya terlalu yakin. TAN mengoptimasi *struktur* model: setiap fitur boleh punya satu fitur lain sebagai parent. Pasangan dipilih dengan maximum weight spanning tree atas conditional mutual information I(X<sub>i</sub>; X<sub>j</sub> | C), yaitu seberapa bergantung dua fitur setelah kelas diketahui. CPT tiap fitur menjadi P(X<sub>i</sub> | C, parent).

### KDB (k-Dependence Bayes)

TAN hanya memberi satu parent fitur. KDB mengizinkan hingga `k` parent: fitur diurutkan dari I(X; C) terbesar, lalu tiap fitur mengambil `k` fitur sebelumnya dengan I(X<sub>i</sub>; X<sub>j</sub> | C) terbesar sebagai parent. Dipakai `BayesNetwork(k=10)`; pemilihan `k` ada di laporan. Untuk `k` besar, CPT yang terlalu besar disimpan sparse (hanya sel yang terisi data).

> Referensi: M. Sahami, "Learning Limited Dependence Bayesian Classifiers," *Proc. KDD-96*, 1996, hlm. 335-338. [https://www.osti.gov/biblio/421307](https://www.osti.gov/biblio/421307)

> Referensi: N. Friedman, D. Geiger, dan M. Goldszmidt, "Bayesian Network Classifiers," *Machine Learning*, vol. 29, no. 2-3, hlm. 131-163, 1997. [https://doi.org/10.1023/A:1007465528199](https://doi.org/10.1023/A:1007465528199)

In [ ]:
bn_variants = {
    "bn_nb": BayesNetwork(n_bins=5),
    "bn_tan": BayesNetwork(n_bins=5, tree=True),
    "bn_kdb": BayesNetwork(n_bins=5, k=10),
}
compare(holdout(bn_variants, X_train, y_train, X_val, y_val, METHOD), kfold(bn_variants, X_train, y_train, METHOD))

### Graf Bayesian Network

Garis putus-putus biru keabuan adalah kelas → fitur (ada untuk semua fitur). Panah merah adalah fitur → fitur, hanya ada pada TAN: panah A → B berarti CPT B dihitung per kombinasi kelas dan nilai A, yaitu P(B | kelas, A). Fitur tanpa panah merah masuk adalah root pohon (node hijau). Arah panah ditentukan oleh root yang dipilih, jadi bukan sebab-akibat.

In [ ]:
from models.bayes_graph import plot_bayes_graph

bn_tan = fit(BayesNetwork(n_bins=5, tree=True), X_train, y_train, METHOD)
fig = plot_bayes_graph(bn_tan, list(X_train.columns), class_name=Settings.TARGET)
fig.savefig(Settings.OUTPUT_DIR / "bayes_graph.png", dpi=160, bbox_inches="tight")
fig

### Cost-sensitive classification

False positive (memprediksi batal, tamu sebenarnya datang) membuat kamar kosong dijual ulang sehingga hotel **overbooked**, dan diasumsikan 2 kali lebih mahal daripada false negative. Penanganan yang berubah hanya **aturan keputusan**, bukan data dan bukan pelatihan. Biaya yang diharapkan memprediksi batal adalah C<sub>FP</sub>(1 - p) dan memprediksi tidak batal adalah C<sub>FN</sub>p, dengan p = P(batal | x). Prediksi batal hanya jika p > C<sub>FP</sub> / (C<sub>FP</sub> + C<sub>FN</sub>) = 2/3, bukan 0,5. Ini diterapkan di `BayesNetwork.predict` lewat `cost_fp` dan `cost_fn`.

> Referensi: C. Elkan, "The Foundations of Cost-Sensitive Learning," *Proc. IJCAI*, 2001, hlm. 973-977. [https://ijcai.org/proceedings/2001/133](https://ijcai.org/proceedings/2001/133)

In [ ]:
from validation import fit_predict, macro_f1

cost_variants = {
    "nb": BayesNetwork(n_bins=5),
    "nb + biaya": BayesNetwork(n_bins=5, cost_fp=2),
    "tan": BayesNetwork(n_bins=5, tree=True),
    "tan + biaya": BayesNetwork(n_bins=5, tree=True, cost_fp=2),
    "kdb": BayesNetwork(n_bins=5, k=10),
    "kdb + biaya": BayesNetwork(n_bins=5, k=10, cost_fp=2),
}

rows = {}
for name, model in cost_variants.items():
    predicted, actual = fit_predict(model, X_train, y_train, X_val, METHOD), y_val.values
    fp, fn = ((predicted == 1) & (actual == 0)).sum(), ((predicted == 0) & (actual == 1)).sum()
    rows[name] = {"FP": fp, "FN": fn, "biaya (2 FP + FN)": 2 * fp + fn, "macro F1": macro_f1(actual, predicted)}
pd.DataFrame(rows).T.astype({"FP": int, "FN": int, "biaya (2 FP + FN)": int})

#### Insights Bonus Bayesian Network

- **TAN lebih baik dari Naive Bayes.** Macro F1 K-Fold 0,760 (std 0,004) berbanding 0,709 (std 0,006). Naive Bayes menghitung fitur yang saling berkaitan (misalnya `total_nights`, `weekday_nights`, `weekend_nights`, atau `month_sin` dan `month_cos`) berulang kali, sedangkan TAN memberi tiap fitur satu parent fitur sehingga bukti ganda itu berkurang.
- **KDB lebih baik lagi.** Dengan sampai 10 parent per fitur, macro F1 K-Fold naik ke 0,812 (std 0,003). Pemilihan `k` ada di laporan; skor turun lagi setelah sekitar k = 11.
- **Cost-sensitive menurunkan biaya total (2 FP + FN):** dari 2463 ke 2114 pada Naive Bayes, 1631 ke 1337 pada TAN, dan 1194 ke 1131 pada KDB. False positive turun dan false negative naik, sesuai harga yang disepakati (FP dua kali lebih mahal).
- **Macro F1 tidak otomatis ikut naik.** Pada TAN turun (0,764 menjadi 0,759) dan pada KDB hampir tetap (0,811 menjadi 0,811), karena F1 menganggap semua kesalahan sama berat. Pada Naive Bayes naik (0,705 menjadi 0,717) karena Naive Bayes semula terlalu sering memprediksi batal.
- Angka di atas berasal dari preprocessing `default` pada notebook ini dengan `seed` tetap. Jika konfigurasinya berubah, jalankan ulang sel di atas dan perbarui angkanya.

## Bonus Optimisasi Logistic Regression

Model `lr` pada bagian Modeling tetap memakai implementasi from-scratch di `src/models/logistic_regression.py`. Eksperimen ini membandingkan batch GD, mini-batch SGD, dan L-BFGS from scratch dengan fitur c010 yang sama. Evaluasi memakai stratified holdout 20% (seed 42), regularisasi L2 0.1, dan threshold 0.42. Cell berikut menampilkan macro F1, accuracy, log-loss, dan status solver, serta menghasilkan tiga CSV prediksi dan `lr_c010_solver_comparison.csv`.

> Referensi L-BFGS: Liu dan Nocedal (1989), *Mathematical Programming* 45, 503-528, [PDF mirror Florida State University](https://people.sc.fsu.edu/~inavon/5420a/liu89limited.pdf). Saratchandran et al. (2023), arXiv:2305.08552, section 3.2, [HTML](https://arxiv.org/html/2305.08552).
> Line search Armijo dan Strong Wolfe: [Wolfram Language Documentation](https://reference.wolframcloud.com/language/tutorial/UnconstrainedOptimizationStepControl.html). Rujukan asli Armijo (1966), *Pacific Journal of Mathematics* 16(1), 1-3, [PDF langsung](https://msp.org/pjm/1966/16-1/pjm-v16-n1-p01-p.pdf).
> Gradient descent dan variannya: [GeeksforGeeks](https://www.geeksforgeeks.org/machine-learning/gradient-descent-algorithm-and-its-variants/) dan [IBM Think](https://www.ibm.com/think/topics/gradient-descent).

In [ ]:
from logistic_submission import run_three_solver_submissions

lr_solver_results, lr_solver_outputs = run_three_solver_submissions()
lr_solver_results[["model", "validation_macro_f1", "validation_accuracy", "validation_log_loss", "iterations", "converged"]]

---

# Deployment <a name="6"></a>

---

Submission menggunakan KNN from scratch dengan konfigurasi `knn_config`. Preprocessor robust di-fit ulang pada seluruh `train.csv`; prediksi disimpan sebagai `output/knn_robust_<timestamp>.csv` bersama model dan preprocessor agar hasilnya dapat direproduksi.

In [ ]:
X_full = train.drop(columns=[Settings.ID, Settings.TARGET])
y_full = train[Settings.TARGET]
prep_final = preps["knn"]
X_full_prepared = prep_final.fit_transform(X_full, y_full)
X_test_prepared = prep_final.transform(test.drop(columns=[Settings.ID]))

submit(
    KNN(**knn_config),
    "knn_robust",
    X_full_prepared,
    y_full,
    X_test_prepared,
    test[Settings.ID],
    METHOD,
    preprocessor=prep_final,
)